# Task 2 · Overfitting
The Task 1 CNN against a larger CNN (1.5× wider convolutions, 2× wider FC layer, 2.9× the parameters), both trained with
the same 5 folds and schedule as in Task 1; then the large CNN with dropout or with weight decay.

In [ ]:
from hw3 import *

B = json.load(open(os.path.join(OUT, 'best.json')))['cnn']
large = dict(model=partial(CNN, 48, 512), lr=B['lr'])
S = {'base': dict(model=partial(CNN, B['arch']), lr=B['lr']), 'large': large,
     'dropout': large | dict(model=partial(CNN, 48, 512, .5)), 'wd': large | dict(wd=.5)}
runs = {'base': B['key'], 'large': 'large', 'dropout': 'large_dropout', 'wd': 'large_wd'}
R = dict(zip(S, cv({runs[k]: S[k] for k in S}).values()))
draw(cnn_layers(48, 512))
save(plt.gcf(), 'arch_large')

Overfitting starts at the first epoch where the generalization loss $GL(t) = 100\,(L_{val}(t) / \min_{s \le t} L_{val}(s) - 1)$
exceeds 5%, i.e. the validation loss is 5% above the best value seen so far (Prechelt's $GL_\alpha$ stopping rule).

In [ ]:
ALPHA = 5
gl = lambda r: 100 * (np.asarray(r['val_loss']) / np.minimum.accumulate(np.asarray(r['val_loss'])) - 1)
onset = lambda r: np.argmax(gl(r) > ALPHA) + 1 if (gl(r) > ALPHA).any() else np.nan
ons = {k: np.array([onset(r) for r in R[k]], float) for k in R}
pd.DataFrame(ons, index=[f'fold {f}' for f in range(FOLDS)])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.3))
G = {'base CNN': ('base', COL['cnn']), 'large CNN': ('large', COL['large'])}
curves({k: (R[n], c) for k, (n, c) in G.items()}, axes[:2])
for k, (n, c) in G.items():
    band(axes[2], [dict(gl=gl(r)) for r in R[n]], 'gl', c, k)
    if np.isfinite(ons[n]).any():
        for ax in axes:
            ax.axvline(np.nanmean(ons[n]), color=c, ls=':', lw=1)
axes[2].axhline(ALPHA, color=INK, ls='--', lw=.8)
axes[2].set(xlabel='epoch', ylabel='generalization loss GL (%)')
save(fig, 'overfit')

In [ ]:
G = {'large CNN': ('large', COL['large']), '+ dropout 0.5': ('dropout', COL['dropout']), '+ weight decay 0.5': ('wd', COL['wd'])}
fig, axes = plt.subplots(2, 2, figsize=(7.2, 4.2))
curves({k: (R[n], c) for k, (n, c) in G.items()}, axes[0])
for k, (n, c) in G.items():
    band(axes[1, 0], R[n], 'wnorm', c, k)
    axes[1, 1].hist(load(runs[n], S[n]['model'])[5].weight.detach().flatten(), 120, histtype='step', color=c, density=True)
axes[1, 0].set(xlabel='epoch', ylabel='norm of all weights')
axes[1, 1].set(xlabel='weight value in the 3072×512 FC layer (fold 1)', ylabel='density', yscale='log')
fig.tight_layout()
save(fig, 'regularize')

In [ ]:
on = lambda k: ms(ons[k], 1) if np.isfinite(ons[k]).all() else f'{np.isfinite(ons[k]).sum()} of 5 folds'
T = pd.DataFrame({k: {'params (M)': f"{R[k][0]['params'] / 1e6:.2f}", 'train acc': ms(col(R[k], 'train_acc')),
                      'val acc': ms(col(R[k], 'val_acc')), 'test acc': ms(col(R[k], 'test_acc')),
                      'gap (pp)': ms(col(R[k], 'train_acc') - col(R[k], 'val_acc')),
                      'best val loss': ms([min(r['val_loss']) for r in R[k]], 1, 3), 'final val loss': ms(col(R[k], 'val_loss'), 1, 3),
                      'onset epoch': on(k)} for k in R}).T
with open(os.path.join(OUT, 't2_table.tex'), 'w') as f:
    f.writelines(' & '.join([k] + list(r)) + ' \\\\\n' for k, r in zip(['Base CNN', 'Large CNN', '+ dropout', '+ weight decay'], T.values))
tex('t2', alpha=ALPHA, **{k + n: v for k in R for n, v in (('Onset', on(k)), ('Gap', T['gap (pp)'][k]),
                                                             ('Test', T['test acc'][k]), ('Loss', T['final val loss'][k]))})
T